# Tree2Code: PySpark Parity Verification

This notebook demonstrates how to verify the numerical consistency (parity) between native models (XGBoost/LightGBM) and the SQL code generated by `tree2code` using PySpark.

In [1]:
import os
import sys
import tempfile
from pathlib import Path

# Set JAVA_HOME
java_home = "/opt/homebrew/opt/openjdk/libexec/openjdk.jdk/Contents/Home"
if os.path.exists(java_home):
    os.environ["JAVA_HOME"] = java_home
    os.environ["PATH"] = f"{java_home}/bin:" + os.environ["PATH"]

import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.datasets import make_classification
from pyspark.sql import SparkSession
from tree2code import convert

# Create a local Spark session with 4G memory
warehouse_dir = tempfile.mkdtemp()
spark = SparkSession.builder \
    .appName("Tree2CodeParityTest") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.sql.warehouse.dir", warehouse_dir) \
    .config("spark.ui.enabled", "false") \
    .getOrCreate()

print(f"Spark Version: {spark.version}")
print(f"JAVA_HOME: {os.environ.get('JAVA_HOME')}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/15 11:09:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Version: 4.1.1
JAVA_HOME: /opt/homebrew/opt/openjdk/libexec/openjdk.jdk/Contents/Home


## 1. Prepare Dataset

In [2]:
X, y = make_classification(
    n_samples=50000, 
    n_features=200, 
    n_informative=5, 
    n_redundant=2, 
    random_state=42
)
feature_names = [f"f{i}" for i in range(200)]
df_train = pd.DataFrame(X, columns=feature_names)

# for col in feature_names[:2]:
#     df_train.loc[np.random.choice(df_train.index, 10), col] = np.nan

print("Sample Data ready.")

Sample Data ready.


## 2. Train Models

In [3]:
xgb_clf = xgb.XGBClassifier(
    n_estimators=200, 
    max_depth=3, 
    learning_rate=0.1, 
    random_state=42, 
    eval_metric='auc'
)
xgb_clf.fit(df_train, y)

lgb_clf = lgb.LGBMClassifier(
    n_estimators=500, 
    num_leaves=16, 
    learning_rate=0.1, 
    random_state=42, 
    verbosity=-1
)
lgb_clf.fit(df_train, y)

print("Simple models trained for parity test.")

Simple models trained for parity test.


## 3. Define Parity Verification Function

In [4]:
def verify_parity(model, model_name, dialect="hive"):
    print(f"\n=== Testing {model_name} ({dialect}) ===")
    
    native_probs = model.predict_proba(df_train)[:, 1]
    
    out = convert(  
        model, 
        to="sql", 
        dialect=dialect, 
        sql_mode="select", 
        table_name="temp_table",
        keep_columns=feature_names,
        literal_format="standard"
    )
    sql_query = out["sql"]["select_sql"]
    
    spark.createDataFrame(df_train).createOrReplaceTempView("temp_table")
    spark_results = spark.sql(sql_query).toPandas()
    
    sql_probs = spark_results["score_p"].values
    
    diff = np.abs(native_probs - sql_probs)
    max_diff = np.max(diff)
    
    threshold = 1e-6 if "xgb" in model_name.lower() else 1e-12
    
    if max_diff < threshold:
        print(f"✅ PASS: Numerical parity achieved! (Max Diff: {max_diff:.2e})")
    else:
        print(f"❌ FAIL: Difference exceeds threshold ({threshold})! (Max Diff: {max_diff:.2e})")

## 4. Run Tests

In [5]:
verify_parity(xgb_clf, "XGBoost")
verify_parity(lgb_clf, "LightGBM")


=== Testing XGBoost (hive) ===


26/04/15 11:10:03 WARN FileSystem: Cannot load filesystem
java.util.ServiceConfigurationError: org.apache.hadoop.fs.FileSystem: Provider org.apache.hadoop.fs.viewfs.ViewFileSystem could not be instantiated
	at java.base/java.util.ServiceLoader.fail(ServiceLoader.java:552)
	at java.base/java.util.ServiceLoader$ProviderImpl.newInstance(ServiceLoader.java:712)
	at java.base/java.util.ServiceLoader$ProviderImpl.get(ServiceLoader.java:672)
	at java.base/java.util.ServiceLoader$2.next(ServiceLoader.java:1256)
	at org.apache.hadoop.fs.FileSystem.loadFileSystems(FileSystem.java:3525)
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3562)
	at org.apache.hadoop.fs.FsUrlStreamHandlerFactory.<init>(FsUrlStreamHandlerFactory.java:77)
	at org.apache.spark.sql.internal.SharedState$.liftedTree2$1(SharedState.scala:209)
	at org.apache.spark.sql.internal.SharedState$.org$apache$spark$sql$internal$SharedState$$setFsUrlStreamHandlerFactory(SharedState.scala:208)
	at org.apache.spark.

✅ PASS: Numerical parity achieved! (Max Diff: 1.19e-07)

=== Testing LightGBM (hive) ===


26/04/15 11:14:29 WARN DAGScheduler: Broadcasting large task binary with size 11.8 MiB
26/04/15 11:14:29 WARN TaskSetManager: Stage 1 contains a task of very large size (7222 KiB). The maximum recommended task size is 1000 KiB.


✅ PASS: Numerical parity achieved! (Max Diff: 2.22e-16)


In [6]:
df_train = df_train.replace(np.nan, None)

In [7]:
verify_parity(xgb_clf, "XGBoost")
verify_parity(lgb_clf, "LightGBM")


=== Testing XGBoost (hive) ===


26/04/15 11:16:05 WARN DAGScheduler: Broadcasting large task binary with size 2.8 MiB
26/04/15 11:16:06 WARN TaskSetManager: Stage 2 contains a task of very large size (7222 KiB). The maximum recommended task size is 1000 KiB.


✅ PASS: Numerical parity achieved! (Max Diff: 1.19e-07)

=== Testing LightGBM (hive) ===


26/04/15 11:16:39 WARN DAGScheduler: Broadcasting large task binary with size 11.8 MiB
26/04/15 11:16:39 WARN TaskSetManager: Stage 3 contains a task of very large size (7222 KiB). The maximum recommended task size is 1000 KiB.


✅ PASS: Numerical parity achieved! (Max Diff: 2.22e-16)


In [8]:
import pickle

In [9]:
with open('./lgb.pkl', 'wb') as f:
    pickle.dump(lgb_clf, f)

In [10]:
df_train.to_parquet('./df_train.pq')

In [11]:
spark.stop()